In [1]:
import pandas as pd
import cupy as cp
import cudf
import cuml
import torch
import gc
import time
from cuml.naive_bayes import BernoulliNB, CategoricalNB, ComplementNB, GaussianNB, MultinomialNB
from cuml.metrics import mean_squared_error, mean_squared_log_error, median_absolute_error, r2_score, accuracy_score, confusion_matrix, kl_divergence
from cuml.metrics import log_loss, roc_auc_score, nan_euclidean_distances, pairwise_distances, sparse_pairwise_distances
from cuml.model_selection import train_test_split, KFold

In [2]:
df = cudf.read_csv('heart_disease_health_indicators_BRFSS2015.csv')
df

,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253675,0.0,1.0,1.0,1.0,45.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,5.0,0.0,1.0,5.0,6.0,7.0
253676,0.0,1.0,1.0,1.0,18.0,0.0,0.0,2.0,0.0,0.0,...,1.0,0.0,4.0,0.0,0.0,1.0,0.0,11.0,2.0,4.0
253677,0.0,0.0,0.0,1.0,28.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0,5.0,2.0
253678,0.0,1.0,0.0,1.0,23.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,1.0,7.0,5.0,1.0


In [3]:
from ipynb.fs.full.Normalization_to_import import *

Execution time: 0.6546282768249512 seconds


In [4]:
from ipynb.fs.full.Data_Generation_to_import import *

Execution time: 0.04196023941040039 seconds


In [5]:
class NaiveBayes(object):
    def __init__(self, dataset, generated_data_global_1):
        self.dataset = dataset.copy().reset_index(drop = True)
        self.X = cudf.DataFrame(self.dataset.copy().drop(['HeartDiseaseorAttack'], axis = 1))
        self.y = cudf.DataFrame(self.dataset['HeartDiseaseorAttack'].copy())
        self.generated_data_global_1 = generated_data_global_1.copy().reset_index(drop = True)
        self.dataset_numpy_array = self.dataset.copy().to_numpy()
        self.X_cupy_array = self.X.to_cupy()
        self.y_cupy_array = self.y.to_cupy()
        self.gen_data_cupy_array = self.generated_data_global_1.to_cupy()
        
        
        
    def train_test_split(self):
        global X_train_global
        global X_test_global
        global y_train_global
        global y_test_global

        X_train, X_test, y_train, y_test = train_test_split(self.X_cupy_array, self.y_cupy_array, test_size=0.2, random_state=42)
        X_train_global = X_train
        X_test_global = X_test
        y_train_global = y_train
        y_test_global = y_test

    def BernoulliNB(self):
        global bnb_global
        global bnb_predict_test_global
        global bnb_predict_global_1
        global mean_squared_error_global_bnb
        global mean_squared_log_error_global_bnb
        global median_absolute_error_global_bnb
        global r2_score_global_bnb
        global accuracy_score_global_bnb
        global kl_divergence_global_bnb
        global log_loss_global_bnb
        global roc_auc_score_global_bnb
        global nan_euclidean_distances_global_bnb
        global pairwise_distances_global_bnb
        global sparse_pairwise_distances_global_bnb

        model = BernoulliNB(alpha=1.0, binarize=0.0, fit_prior=True, class_prior=None, output_type=None, verbose=False)
        bnb = model.fit(X_train_global.copy(), y_train_global.copy().ravel())
        bnb_global = bnb
        
        preds_test = model.predict(X_test_global.copy())
        bnb_predict_test_global = cudf.DataFrame(preds_test.copy(), columns = ['HeartDiseaseorAttack']).reset_index(drop = True)
        
        mean_squared_error_global_bnb = mean_squared_error(y_test_global.copy(), preds_test)
        #mean_squared_log_error_global_bnb = mean_squared_log_error(y_test_global.copy(), preds_test)
        median_absolute_error_global_bnb = median_absolute_error(y_test_global.copy(), preds_test)
        r2_score_global_bnb = r2_score(y_test_global.copy(), preds_test)
        accuracy_score_global_bnb = accuracy_score(y_test_global.copy(), preds_test)
        kl_divergence_global_bnb = kl_divergence(y_test_global.copy(), preds_test)
        #log_loss_global_bnb = log_loss(y_test_global.copy(), preds_test)
        roc_auc_score_global_bnb = roc_auc_score(y_test_global.copy(), preds_test)
        #nan_euclidean_distances_global_bnb = nan_euclidean_distances(y_test_global.copy(), preds_test)
        #pairwise_distances_global_bnb = pairwise_distances(y_test_global.copy(), preds_test)
        #sparse_pairwise_distances_global_bnb = sparse_pairwise_distances(y_test_global.copy(), preds_test)
     
        preds_1 = model.predict(self.gen_data_cupy_array)
        bnb_predict_global_1 = cudf.DataFrame(preds_1.copy(), columns = ['HeartDiseaseorAttack']).reset_index(drop = True)

    def CategoricalNB(self):
        global cnb_global
        global cnb_predict_test_global
        global cnb_predict_global_1
        global mean_squared_error_global_cnb
        global mean_squared_log_error_global_cnb
        global median_absolute_error_global_cnb
        global r2_score_global_cnb
        global accuracy_score_global_cnb
        global kl_divergence_global_cnb
        global log_loss_global_cnb
        global roc_auc_score_global_cnb
        global nan_euclidean_distances_global_cnb
        global pairwise_distances_global_cnb
        global sparse_pairwise_distances_global_cnb

        model = CategoricalNB(alpha=1.0, fit_prior=True, class_prior=None, output_type=None, verbose=False)
        cnb = model.fit(X_train_global.copy(), y_train_global.copy().ravel())
        cnb_global = cnb
        
        preds_test = model.predict(X_test_global.copy())
        cnb_predict_test_global = cudf.DataFrame(preds_test.copy(), columns = ['HeartDiseaseorAttack']).reset_index(drop = True)
        
        mean_squared_error_global_cnb = mean_squared_error(y_test_global.copy(), preds_test)
        #mean_squared_log_error_global_cnb = mean_squared_log_error(y_test_global.copy(), preds_test)
        median_absolute_error_global_cnb = median_absolute_error(y_test_global.copy(), preds_test)
        r2_score_global_cnb = r2_score(y_test_global.copy(), preds_test)
        accuracy_score_global_cnb = accuracy_score(y_test_global.copy(), preds_test)
        kl_divergence_global_cnb = kl_divergence(y_test_global.copy(), preds_test)
        #log_loss_global_cnb = log_loss(y_test_global.copy(), preds_test)
        roc_auc_score_global_cnb = roc_auc_score(y_test_global.copy(), preds_test)
        #nan_euclidean_distances_global_cnb = nan_euclidean_distances(y_test_global.copy(), preds_test)
        #pairwise_distances_global_cnb = pairwise_distances(y_test_global.copy(), preds_test)
        #sparse_pairwise_distances_global_cnb = sparse_pairwise_distances(y_test_global.copy(), preds_test)
     
        preds_1 = model.predict(self.gen_data_cupy_array)
        cnb_predict_global_1 = cudf.DataFrame(preds_1.copy(), columns = ['HeartDiseaseorAttack']).reset_index(drop = True)

    def ComplementNB(self):
        global comnb_global
        global comnb_predict_test_global
        global comnb_predict_global_1
        global mean_squared_error_global_comnb
        global mean_squared_log_error_global_comnb
        global median_absolute_error_global_comnb
        global r2_score_global_comnb
        global accuracy_score_global_comnb
        global kl_divergence_global_comnb
        global log_loss_global_comnb
        global roc_auc_score_global_comnb
        global nan_euclidean_distances_global_comnb
        global pairwise_distances_global_comnb
        global sparse_pairwise_distances_global_comnb

        model = ComplementNB(alpha=1.0, fit_prior=True, class_prior=None, norm=False, output_type=None, verbose=False)
        comnb = model.fit(X_train_global.copy(), y_train_global.copy().ravel())
        comnb_global = comnb
        
        preds_test = model.predict(X_test_global.copy())
        comnb_predict_test_global = cudf.DataFrame(preds_test.copy(), columns = ['HeartDiseaseorAttack']).reset_index(drop = True)
        
        mean_squared_error_global_comnb = mean_squared_error(y_test_global.copy(), preds_test)
        #mean_squared_log_error_global_comnb = mean_squared_log_error(y_test_global.copy(), preds_test)
        median_absolute_error_global_comnb = median_absolute_error(y_test_global.copy(), preds_test)
        r2_score_global_comnb = r2_score(y_test_global.copy(), preds_test)
        accuracy_score_global_comnb = accuracy_score(y_test_global.copy(), preds_test)
        kl_divergence_global_comnb = kl_divergence(y_test_global.copy(), preds_test)
        #log_loss_global_comnb = log_loss(y_test_global.copy(), preds_test)
        roc_auc_score_global_comnb = roc_auc_score(y_test_global.copy(), preds_test)
        #nan_euclidean_distances_global_comnb = nan_euclidean_distances(y_test_global.copy(), preds_test)
        #pairwise_distances_global_comnb = pairwise_distances(y_test_global.copy(), preds_test)
        #sparse_pairwise_distances_global_comnb = sparse_pairwise_distances(y_test_global.copy(), preds_test)
     
        preds_1 = model.predict(self.gen_data_cupy_array)
        comnb_predict_global_1 = cudf.DataFrame(preds_1.copy(), columns = ['HeartDiseaseorAttack']).reset_index(drop = True)

    def GaussianNB(self):
        global gnb_global
        global gnb_predict_test_global
        global gnb_predict_global_1
        global mean_squared_error_global_gnb
        global mean_squared_log_error_global_gnb
        global median_absolute_error_global_gnb
        global r2_score_global_gnb
        global accuracy_score_global_gnb
        global kl_divergence_global_gnb
        global log_loss_global_gnb
        global roc_auc_score_global_gnb
        global nan_euclidean_distances_global_gnb
        global pairwise_distances_global_gnb
        global sparse_pairwise_distances_global_gnb

        model = GaussianNB(priors=None, var_smoothing=1e-09, output_type=None, verbose=False)
        gnb = model.fit(X_train_global.copy(), y_train_global.copy().ravel())
        gnb_global = gnb
        
        preds_test = model.predict(X_test_global.copy())
        gnb_predict_test_global = cudf.DataFrame(preds_test.copy(), columns = ['HeartDiseaseorAttack']).reset_index(drop = True)
        
        mean_squared_error_global_gnb = mean_squared_error(y_test_global.copy(), preds_test)
        #mean_squared_log_error_global_gnb = mean_squared_log_error(y_test_global.copy(), preds_test)
        median_absolute_error_global_gnb = median_absolute_error(y_test_global.copy(), preds_test)
        r2_score_global_gnb = r2_score(y_test_global.copy(), preds_test)
        accuracy_score_global_gnb = accuracy_score(y_test_global.copy(), preds_test)
        kl_divergence_global_gnb = kl_divergence(y_test_global.copy(), preds_test)
        #log_loss_global_gnb = log_loss(y_test_global.copy(), preds_test)
        roc_auc_score_global_gnb = roc_auc_score(y_test_global.copy(), preds_test)
        #nan_euclidean_distances_global_gnb = nan_euclidean_distances(y_test_global.copy(), preds_test)
        #pairwise_distances_global_gnb = pairwise_distances(y_test_global.copy(), preds_test)
        #sparse_pairwise_distances_global_gnb = sparse_pairwise_distances(y_test_global.copy(), preds_test)
     
        preds_1 = model.predict(self.gen_data_cupy_array)
        gnb_predict_global_1 = cudf.DataFrame(preds_1.copy(), columns = ['HeartDiseaseorAttack']).reset_index(drop = True)

    def MultinomialNB(self):
        global mnb_global
        global mnb_predict_test_global
        global mnb_predict_global_1
        global mean_squared_error_global_mnb
        global mean_squared_log_error_global_mnb
        global median_absolute_error_global_mnb
        global r2_score_global_mnb
        global accuracy_score_global_mnb
        global kl_divergence_global_mnb
        global log_loss_global_mnb
        global roc_auc_score_global_mnb
        global nan_euclidean_distances_global_mnb
        global pairwise_distances_global_mnb
        global sparse_pairwise_distances_global_mnb

        model = MultinomialNB(alpha=1.0, fit_prior=True, class_prior=None, verbose=False, output_type=None)
        mnb = model.fit(X_train_global.copy(), y_train_global.copy().ravel())
        mnb_global = mnb
        
        preds_test = model.predict(X_test_global.copy())
        mnb_predict_test_global = cudf.DataFrame(preds_test.copy(), columns = ['HeartDiseaseorAttack']).reset_index(drop = True)
        
        mean_squared_error_global_mnb = mean_squared_error(y_test_global.copy(), preds_test)
        #mean_squared_log_error_global_mnb = mean_squared_log_error(y_test_global.copy(), preds_test)
        median_absolute_error_global_mnb = median_absolute_error(y_test_global.copy(), preds_test)
        r2_score_global_mnb = r2_score(y_test_global.copy(), preds_test)
        accuracy_score_global_mnb = accuracy_score(y_test_global.copy(), preds_test)
        kl_divergence_global_mnb = kl_divergence(y_test_global.copy(), preds_test)
        #log_loss_global_mnb = log_loss(y_test_global.copy(), preds_test)
        roc_auc_score_global_mnb = roc_auc_score(y_test_global.copy(), preds_test)
        #nan_euclidean_distances_global_mnb = nan_euclidean_distances(y_test_global.copy(), preds_test)
        #pairwise_distances_global_mnb = pairwise_distances(y_test_global.copy(), preds_test)
        #sparse_pairwise_distances_global_mnb = sparse_pairwise_distances(y_test_global.copy(), preds_test)
     
        preds_1 = model.predict(self.gen_data_cupy_array)
        mnb_predict_global_1 = cudf.DataFrame(preds_1.copy(), columns = ['HeartDiseaseorAttack']).reset_index(drop = True)

    def main(self):
        st = time.time()
        self.train_test_split()
        self.BernoulliNB()
        self.CategoricalNB()
        self.ComplementNB()
        self.GaussianNB()
        self.MultinomialNB()
        et = time.time()
        elapsed_time = et - st
        print('Execution time:', elapsed_time, 'seconds')

In [6]:
class Metrics(object):
    def BernoulliNB(self):
        print('BernoulliNB: ')
        print('Mean squared error: ')
        print(mean_squared_error_global_bnb)
        print('\n')
        print('Median Absolute Error: ')
        print(median_absolute_error_global_bnb)
        print('\n')
        print('R2 Score: ')
        print(r2_score_global_bnb)
        print('\n')
        print('Accuracy Score: ')
        print(accuracy_score_global_bnb)
        print('\n')
        print('Kl Divergence: ')
        print(kl_divergence_global_bnb)
        print('\n')
        print('ROC AUC Score: ')
        print(roc_auc_score_global_bnb)
        print('\n')

    def CategoricalNB(self):
        print('CategoricalNB: ')
        print('Mean squared error: ')
        print(mean_squared_error_global_cnb)
        print('\n')
        print('Median Absolute Error: ')
        print(median_absolute_error_global_cnb)
        print('\n')
        print('R2 Score: ')
        print(r2_score_global_cnb)
        print('\n')
        print('Accuracy Score: ')
        print(accuracy_score_global_cnb)
        print('\n')
        print('Kl Divergence: ')
        print(kl_divergence_global_cnb)
        print('\n')
        print('ROC AUC Score: ')
        print(roc_auc_score_global_cnb)
        print('\n')

    def ComplementNB(self):
        print('ComplementNB: ')
        print('Mean squared error: ')
        print(mean_squared_error_global_comnb)
        print('\n')
        print('Median Absolute Error: ')
        print(median_absolute_error_global_comnb)
        print('\n')
        print('R2 Score: ')
        print(r2_score_global_comnb)
        print('\n')
        print('Accuracy Score: ')
        print(accuracy_score_global_comnb)
        print('\n')
        print('Kl Divergence: ')
        print(kl_divergence_global_comnb)
        print('\n')
        print('ROC AUC Score: ')
        print(roc_auc_score_global_comnb)
        print('\n')

    def GaussianNB(self):
        print('GaussianNB: ')
        print('Mean squared error: ')
        print(mean_squared_error_global_gnb)
        print('\n')
        print('Median Absolute Error: ')
        print(median_absolute_error_global_gnb)
        print('\n')
        print('R2 Score: ')
        print(r2_score_global_gnb)
        print('\n')
        print('Accuracy Score: ')
        print(accuracy_score_global_gnb)
        print('\n')
        print('Kl Divergence: ')
        print(kl_divergence_global_gnb)
        print('\n')
        print('ROC AUC Score: ')
        print(roc_auc_score_global_gnb)
        print('\n')

    def MultinomialNB(self):
        print('MultinomialNB: ')
        print('Mean squared error: ')
        print(mean_squared_error_global_mnb)
        print('\n')
        print('Median Absolute Error: ')
        print(median_absolute_error_global_mnb)
        print('\n')
        print('R2 Score: ')
        print(r2_score_global_mnb)
        print('\n')
        print('Accuracy Score: ')
        print(accuracy_score_global_mnb)
        print('\n')
        print('Kl Divergence: ')
        print(kl_divergence_global_mnb)
        print('\n')
        print('ROC AUC Score: ')
        print(roc_auc_score_global_mnb)
        print('\n')

    def main(self):
        self.BernoulliNB()
        self.CategoricalNB()
        self.ComplementNB()
        self.GaussianNB()
        self.MultinomialNB()


        

In [7]:
nb = NaiveBayes(df, generated_data_global)
nb.main()
metrics_naive_bayes = Metrics()
print("Metrics for not normalized data is ready. Run 'metrics_naive_bayes.main()'!")
#metrics_naive_bayes.main()

Execution time: 0.3035132884979248 seconds
Metrics for not normalized data is ready. Run 'metrics_naive_bayes.main()'!


/home/xy/Desktop/ml/rapids-cuml/lib/python3.10/site-packages/cuml/naive_bayes/naive_bayes.py:1563: UserWarning: X dtype is not int32. X will be converted, which will increase memory consumption
  warnings.warn(
/home/xy/Desktop/ml/rapids-cuml/lib/python3.10/site-packages/cuml/naive_bayes/naive_bayes.py:1589: UserWarning: X dtype is not int32. X will be converted, which will increase memory consumption
  warnings.warn(


In [8]:
torch.cuda.empty_cache()
gc.collect()

0

In [9]:
nb_maxAbsScaler = NaiveBayes(maxAbsScaler_merged_global, generated_data_global)
nb_maxAbsScaler.main()
metrics_MaxAbsScaler_naive_bayes = Metrics()
print("Metrics for MaxAbsScaler normalized data is ready. Run 'metrics_MaxAbsScaler_naive_bayes.main()'!")
#metrics_MaxAbsScaler_naive_bayes.main()

Execution time: 0.2646510601043701 seconds
Metrics for MaxAbsScaler normalized data is ready. Run 'metrics_MaxAbsScaler_naive_bayes.main()'!


In [10]:
torch.cuda.empty_cache()
gc.collect()

0

In [11]:
nb_MinMaxScaler = NaiveBayes(MinMaxScaler_merged_global, generated_data_global)
nb_MinMaxScaler.main()
metrics_MinMaxScaler_naive_bayes= Metrics()
print("Metrics for MinMaxScaler normalized data is ready. Run 'metrics_MinMaxScaler_naive_bayes.main()'!")
#metrics_MinMaxScaler_naive_bayes.main()

Execution time: 0.1965041160583496 seconds
Metrics for MinMaxScaler normalized data is ready. Run 'metrics_MinMaxScaler_naive_bayes.main()'!


In [12]:
torch.cuda.empty_cache()
gc.collect()

0

In [13]:
nb_normalizer = NaiveBayes(normalizer_merged_global, generated_data_global)
nb_normalizer.main()
metrics_Normalizer_naive_bayes = Metrics()
print("Metrics for Normalizer normalized data is ready. Run 'metrics_Normalizer_naive_bayes.main()'!")
#metrics_Normalizer_naive_bayes.main()

Execution time: 0.18510198593139648 seconds
Metrics for Normalizer normalized data is ready. Run 'metrics_Normalizer_naive_bayes.main()'!


In [14]:
torch.cuda.empty_cache()
gc.collect()

0

In [15]:
nb_binarizer = NaiveBayes(binarizer_merged_global, generated_data_global)
nb_binarizer.main()
metrics_Binarizer_naive_bayes = Metrics()
print("Metrics for Binarizer normalized data is ready. Run 'metrics_Binarizer_naive_bayes.main()'!")
#metrics_Binarizer_naive_bayes.main()

Execution time: 0.19516873359680176 seconds
Metrics for Binarizer normalized data is ready. Run 'metrics_Binarizer_naive_bayes.main()'!


In [16]:
torch.cuda.empty_cache()
gc.collect()

0

In [17]:
nb_function_transformer = NaiveBayes(function_transformer_merged_global, generated_data_global)
nb_function_transformer.main()
metrics_FunctionTransformer_naive_bayes = Metrics()
print("Metrics for FunctionTransformer normalized data is ready. Run 'metrics_FunctionTransformer_naive_bayes.main()'!")
#metrics_FunctionTransformer_naive_bayes.main()

Execution time: 0.20185017585754395 seconds
Metrics for FunctionTransformer normalized data is ready. Run 'metrics_FunctionTransformer_naive_bayes.main()'!


In [18]:
torch.cuda.empty_cache()
gc.collect()

0

In [19]:
nb_KBinsDiscretizer = NaiveBayes(KBinsDiscretizer_merged_global, generated_data_global)
nb_KBinsDiscretizer.main()
metrics_KBinsDiscretizer_naive_bayes = Metrics()
print("Metrics for KBinsDiscretizer normalized data is ready. Run 'metrics_KBinsDiscretizer_naive_bayes.main()'!")
#metrics_KBinsDiscretizer_naive_bayes.main()

Execution time: 0.20641350746154785 seconds
Metrics for KBinsDiscretizer normalized data is ready. Run 'metrics_KBinsDiscretizer_naive_bayes.main()'!


In [20]:
torch.cuda.empty_cache()
gc.collect()

0